In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"D:\SecurePay-AI\dataset\cleaned\transactions_clean.csv")

In [2]:
df.head()

,step,type,amount,nameorig,oldbalanceorg,newbalanceorig,namedest,oldbalancedest,newbalancedest,isfraud,isflaggedfraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [3]:
# transaction features 

# feature 1 : transaction amount category

df["amount_category"]=pd.cut(
    df["amount"],
    bins=[0,1000,10000,50000,100000,1000000,np.inf],
    labels=[
        "Very Low",
        "Low",
        "Medium",
        "High",
        "Very High",
        "Extreme"
    ]
)

In [4]:
# feature 2 : large transaction flag

df["large_transaction"]=(
    df["amount"]>100000
).astype(int)

In [6]:
# sender feature
# sender transaction count

sender_count =(
    df.groupby("nameorig").size()
)

df["sender_transaction_count"] = (
    df["nameorig"]
      .map(sender_count)
)

In [7]:
# sender total amount 

sender_amount = (
    df.groupby("nameorig")["amount"].sum()
)

df["sender_transaction_amount"]=(
    df["nameorig"].map(sender_amount)
)

In [11]:
# receiver feature

# receiver transaction count 

receiver_count = (
    df.groupby("namedest").size()
)

df["receiver_transaction_count"]=(
    df["namedest"].map(receiver_count)
)

receiver_count

namedest
C1000004082     6
C1000004940    13
C1000013769    13
C100001587      9
C1000015936    16
               ..
M999998692      1
M99999900       1
M999999089      1
M999999543      1
M999999784      1
Length: 2722362, dtype: int64

In [10]:
# receiver total amount 

receiver_amount =(
    df.groupby("namedest")["amount"].sum()
)

df["receiver_total_amount"]=(
    df["namedest"].map(receiver_amount)
)

receiver_amount

namedest
C1000004082    2259324.39
C1000004940    2534004.05
C1000013769    6204082.94
C100001587     1404313.66
C1000015936    2267960.19
                  ...    
M999998692        3156.54
M99999900        34263.71
M999999089       16725.52
M999999543       19365.23
M999999784        2327.35
Name: amount, Length: 2722362, dtype: float64

In [12]:
# receiver fraud count 

receiver_fraud =(
    df[df["isfraud"]==1].groupby("namedest").size()
)
df["receiver_fraud_count"]=(
    df["namedest"].map(receiver_fraud).fillna(0)
)

receiver_fraud

namedest
C1000039615    1
C1000367306    1
C1000407130    1
C1000836187    1
C1000855680    1
              ..
C998514614     1
C999409522     1
C999470580     1
C999708230     1
C999955448     1
Length: 8169, dtype: int64

In [13]:
# receiver fraud rate

df["receiver_fraud_rate"]=(
    df["receiver_fraud_count"]/df["receiver_transaction_count"]
)

In [15]:
df["sender_balance_change"] = (
    df["oldbalanceorg"] -
    df["newbalanceorig"]
)

In [16]:
# receiver balance difference 
df["receiver_balance_change"]=(
    df["newbalancedest"]-
    df["oldbalancedest"]
)

In [17]:
# balance ratio 
df["balance_ratio"]=(
    df["amount"]/
    (df["oldbalanceorg"]+1)
)

In [19]:
# time feature 
# day number

df["day"]=(
    df["step"]//24
)

print(df["day"])

0           0
1           0
2           0
3           0
4           0
           ..
6362615    30
6362616    30
6362617    30
6362618    30
6362619    30
Name: day, Length: 6362620, dtype: int64


In [20]:
#hour 
df["hour"]=(
    df["step"]%24
)

In [21]:
# night transaction flag

df["night_transaction"]=(
    df["hour"].between(0,5)
).astype(int)

In [22]:
# receiver risk score 

df["receiver_risk_score"]=(
    df["receiver_fraud_rate"]*60
    +(df["receiver_transaction_count"]/df["receiver_transaction_count"].max())*20
    +(df["receiver_total_amount"]/df["receiver_total_amount"].max())*20
)

In [27]:
df["receiver_risk_level"] = pd.cut(
    df["receiver_risk_score"],
    bins=[0,20,40,60,80,100],
    labels=[
        "Very Low",
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

In [28]:
# view final dataset

df.head()

,step,type,amount,nameorig,oldbalanceorg,newbalanceorig,namedest,oldbalancedest,newbalancedest,isfraud,...,receiver_fraud_count,receiver_fraud_rate,sender_balance_change,receiver_balance_change,balance_ratio,day,hour,night_transaction,receiver_risk_score,receiver_risk_level
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,...,0.0,0.000000,9839.64,0.0,0.057834,0,1,1,0.177542,Very Low
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,...,0.0,0.000000,1864.28,0.0,0.087731,0,1,1,0.177095,Very Low
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,...,1.0,0.022727,181.00,0.0,0.994505,0,1,1,9.706460,Very Low
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,...,1.0,0.024390,181.00,-21182.0,0.994505,0,1,1,9.728175,Very Low
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,...,0.0,0.000000,11668.14,0.0,0.280788,0,1,1,0.177644,Very Low


In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 27 columns):
 #   Column                      Dtype   
---  ------                      -----   
 0   step                        int64   
 1   type                        object  
 2   amount                      float64 
 3   nameorig                    object  
 4   oldbalanceorg               float64 
 5   newbalanceorig              float64 
 6   namedest                    object  
 7   oldbalancedest              float64 
 8   newbalancedest              float64 
 9   isfraud                     int64   
 10  isflaggedfraud              int64   
 11  amount_category             category
 12  large_transaction           int64   
 13  sender_transaction_count    int64   
 14  sender_transaction_amount   float64 
 15  receiver_transaction_count  int64   
 16  receiver_total_amount       float64 
 17  receiver_fraud_count        float64 
 18  receiver_fraud_rate         float64 
 19  

In [30]:
df.columns

Index(['step', 'type', 'amount', 'nameorig', 'oldbalanceorg', 'newbalanceorig',
       'namedest', 'oldbalancedest', 'newbalancedest', 'isfraud',
       'isflaggedfraud', 'amount_category', 'large_transaction',
       'sender_transaction_count', 'sender_transaction_amount',
       'receiver_transaction_count', 'receiver_total_amount',
       'receiver_fraud_count', 'receiver_fraud_rate', 'sender_balance_change',
       'receiver_balance_change', 'balance_ratio', 'day', 'hour',
       'night_transaction', 'receiver_risk_score', 'receiver_risk_level'],
      dtype='object')

In [32]:
df.to_csv(r"D:\SecurePay-AI\dataset\feature_engineered\feature_engineered.csv",index=False)